# Calculating recycler crafts for RNG skipping

To skip forwards in-game, I make use of scrap recycling. It consumes 12 RNG calls per craft result. I want to know how many crafts I need to perform spread over `n` machines to hit a specific number of calls.

Note that we may require fewer input items due to productivity, as any craft *completion* calls the RNG, and not the craft beginnings.

This notebook contains the logic behind that calculation, generated using integer math only to be translatable into combinators. Since there are multiple steps of rounding involved, I could not figure this out by hand and had to simulate it first.


## Simulation
First a simple simulation of the recycling process, which will be used to verify the loop-less version later on.

In [1]:
def calc_num_crafts_stupid(
    target_calls: int,
    num_machines: int,
    prod_10x: int,
    res_per_craft: int = 12,
) -> tuple[int, int, int]:
    """Computes how many crafts are required to exactly hit target calls."""
    # How many items do all the recyclers get?
    made = 0
    state = 0
    all_crafts = 0
    while True:
        next_state = state + 10 + prod_10x
        next_made = made + (next_state // 10) * res_per_craft * num_machines
        if next_made > target_calls:
            break
        state = next_state % 10
        made = next_made
        all_crafts += 1

    # How many of the recyclers get an additional item?
    per_next_craft = (next_state // 10) * res_per_craft
    single_crafts = 0
    while True:
        next_made = made + per_next_craft
        if next_made > target_calls:
            break
        made = next_made
        single_crafts += 1
    assert 0 <= single_crafts < num_machines, "Each machine should at most get one additional item"

    # How many items remain? These can be split over all machines because there is no productivity.
    remaining = target_calls - made
    return all_crafts, single_crafts, remaining

In [2]:
def compute_num_calls(
    crafts: tuple[int, int, int],
    num_machines: int,
    prod_10x: int,
    res_per_craft: int = 12,
) -> int:
    all_crafts, single_crafts, remaining = crafts
    state = all_crafts * (10 + prod_10x)
    calls = (state // 10) * res_per_craft * num_machines
    state = (state % 10) + (10 + prod_10x)
    calls += single_crafts * (state // 10) * res_per_craft
    calls += remaining
    return calls

## Loop-less calculation

A loop-less version of the above computation generated by gemini.

In [3]:
def calc_num_crafts_optimized(
    target_calls: int,
    num_machines: int,
    prod_10x: int,
    res_per_craft: int = 12,
) -> tuple[int, int, int]:
    """Computes how many crafts are required using only integer math."""
    p = 10 + prod_10x
    items_per_full_step = res_per_craft * num_machines

    # 1. Solve for all_crafts (n).
    # We need the largest n such that: (n * p // 10) * items_per_full_step <= target_calls.
    # Let Q = target_calls // items_per_full_step.
    # We solve n * p // 10 <= Q, which is n <= (10 * Q + 9) // p.
    q = target_calls // items_per_full_step
    all_crafts = (10 * q + 9) // p

    # 2. Calculate items made after all_crafts.
    made_by_all = (all_crafts * p // 10) * items_per_full_step

    # 3. Solve for single_crafts (s).
    # This looks at the productivity of the very next craft (n + 1).
    # The jump in productivity is floor((n + 1) * p / 10) - floor(n * p / 10).
    next_total_p = (all_crafts + 1) * p // 10
    current_total_p = all_crafts * p // 10
    per_next_craft = (next_total_p - current_total_p) * res_per_craft

    # Because of how all_crafts is calculated, this is guaranteed to be < num_machines.
    single_crafts = (target_calls - made_by_all) // per_next_craft

    # 4. Calculate remaining calls.
    made_final = made_by_all + single_crafts * per_next_craft
    remaining = target_calls - made_final

    return all_crafts, single_crafts, remaining

## Combinator translation

The same loop-less calculation, now organized by circuit-network tick.

In [4]:
def calc_num_crafts_combinator_optimized(
    target_calls: int,
    num_machines: int,
    prod_10x: int,
    res_per_craft: int = 12,
) -> tuple[int, int, int]:
    # Preparation not factored in.
    Full = num_machines * res_per_craft
    Prod = prod_10x + 10

    # TICK 1: Calculate base cycles.
    Q = target_calls // Full

    # TICK 2: Numerator for craft calculation.
    # Factorio: Input Q, Output Q * 10 + 9.
    A = (Q * 10) + 9

    # TICK 3: Final all_crafts (n).
    all_crafts = A // Prod

    # TICK 4: Calculate total productivity potential.
    # This is the np value needed for subsequent steps.
    np_val = all_crafts * Prod

    # TICK 5: Parallel breakdown of np.
    rem10 = np_val % 10
    p_all = np_val // 10

    # TICK 6: Parallel calculation of items and next-step productivity.
    # p_inc is the productivity multiplier of the very next craft cycle.
    p_inc = (rem10 + Prod) // 10
    made_all = p_all * Full

    # TICK 7: Parallel calculation of remaining gap and items per single craft.
    R = target_calls - made_all
    Is = p_inc * res_per_craft

    # TICK 8: Final outputs.
    single_crafts = R // Is
    remaining = R % Is

    return all_crafts, single_crafts, remaining

## Verification

Check the combinator-oriented calculation against the original simulation.

In [7]:
res_per_craft = 12
for prod_10x in range(20):
    for num_machines in range(1, 11):
        for calls in range(2000):
            crafts = calc_num_crafts_stupid(calls, num_machines, prod_10x, res_per_craft)
            crafts_opt = calc_num_crafts_combinator_optimized(calls, num_machines, prod_10x, res_per_craft)
            assert compute_num_calls(crafts, num_machines, prod_10x, res_per_craft) == calls
            assert crafts == crafts_opt

print("Verified all combinations.")

Verified all combinations.
